# PS S6E6 - 010 shared002 LightGBM as-is

Experiment: `exp_20260604_010_shared002_lgb_as_is`

Purpose:
- Reproduce only the LightGBM part from `shared002.ipynb`.
- Keep shared002 feature engineering as close to as-is as possible.
- Do not run RealMLP.
- Do not run shared002 Hill Climbing.
- Save OOF / pred `.npy` with the full `exp_*` id in the filename.
- Check whether this FE-rich LGB produces a useful low-correlation material against 007.

Source code intent:
- shared002 originally trains RealMLP + LightGBM + Hill Climbing.
- This notebook extracts the LightGBM part only.
- The original shared002 LightGBM OOF was reported around `0.96251`.

Important:
- `TargetEncoder` is fit inside each fold using only `X_tr, y_tr`.
- `feature_engineering()` is fit on full train, but it uses non-target transforms only:
  factorization, floor categories, KBinsDiscretizer, and category combinations.

In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import lightgbm as lgb

warnings.filterwarnings("ignore")


try:
    import yaml
except Exception:
    yaml = None


COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260604_010_shared002_lgb_as_is"
SEED = 42
FOLDS = 5

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID = "id"
TARGET = "class"

CLASS_TO_ID = {"GALAXY": 0, "QSO": 1, "STAR": 2}
ID_TO_CLASS = {0: "GALAXY", 1: "QSO", 2: "STAR"}
CLASS_NAMES = ["GALAXY", "QSO", "STAR"]

# Required naming rule: include exp_* id in oof/pred npy filenames.
OOF_NPY = OUTDIR / "oof_exp_20260604_010_shared002_lgb_as_is_proba.npy"
PRED_NPY = OUTDIR / "pred_exp_20260604_010_shared002_lgb_as_is_proba.npy"

def seed_everything(seed: int):
    np.random.seed(seed)
    random.seed(seed)

seed_everything(SEED)


def metric(y_true, y_pred_proba):
    return balanced_accuracy_score(y_true, np.argmax(y_pred_proba, axis=1))


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


print("LightGBM version:", lgb.__version__)
print("OUTDIR:", OUTDIR)
print("OOF_NPY:", OOF_NPY)
print("PRED_NPY:", PRED_NPY)

LightGBM version: 4.6.0
OUTDIR: /kaggle/working/exp_20260604_010_shared002_lgb_as_is
OOF_NPY: /kaggle/working/exp_20260604_010_shared002_lgb_as_is/oof_exp_20260604_010_shared002_lgb_as_is_proba.npy
PRED_NPY: /kaggle/working/exp_20260604_010_shared002_lgb_as_is/pred_exp_20260604_010_shared002_lgb_as_is_proba.npy


In [2]:
# ============================================================
# 1. Load data
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

print("train:", train.shape)
print("test :", test.shape)
print("sample:", sample.shape)
print("train columns:", train.columns.tolist())
print("test columns :", test.columns.tolist())
print("sample columns:", sample.columns.tolist())

train[TARGET] = train[TARGET].map(CLASS_TO_ID).astype(int)

X = train.drop([ID, TARGET], axis=1)
y = train[TARGET].copy()
X_test = test.drop([ID], axis=1)
test_id = test[ID].copy()

display(X.head())
display(X_test.head())

print("target distribution:")
display(y.value_counts().sort_index().rename(index=ID_TO_CLASS).to_frame("count"))

train: (577347, 12)
test : (247435, 11)
sample: (247435, 2)
train columns: ['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'class']
test columns : ['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population']
sample columns: ['id', 'class']


,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence


,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


target distribution:


,count
class,
GALAXY,377480
QSO,117143
STAR,82724


In [3]:
# ============================================================
# 2. shared002 feature engineering
# ============================================================

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

category_map = {}
important_combos = [("alpha_cat_", "delta_cat_"), ("u_cat_", "z_cat_")]
important_combos = sorted(important_combos)

def feature_engineering(df, fit=False):
    df = df.copy()

    # shared002 as-is ratio features.
    df["_g_/_redshift"] = (df["g"] / (df["redshift"] + 1e-6)).astype("float32")
    df["_i_/_redshift"] = (df["i"] / (df["redshift"] + 1e-6)).astype("float32")

    # Original object categorical columns.
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype("int32")
        df[col] = codes
        df[col] = df[col].astype("category")

    # Floor-based numeric categorical features.
    for col in num_cols:
        cat_name = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype("int32")
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype("category")

    # Binning features.
    bin_config = {"delta": [100, 500]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ["quantile"]:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode="ordinal",
                        strategy=strategy,
                        subsample=None,
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype("int32")
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype("int32")
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype("category")

    # Combo categorical features.
    combo_names = []
    for cols in important_combos:
        combo_name = "_".join(cols) + "_"
        combo_names.append(combo_name)

        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + "_" + df[col].astype(str)

        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype("int32")

        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype("category")

    new_cat_cols = [c for c in df.columns if c.endswith("_")]
    new_num_cols = [c for c in df.columns if c.startswith("_")]
    return df, new_cat_cols, new_num_cols, combo_names


X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)

cat_cols += new_cat_cols
num_cols += new_num_cols

cat_cols = sorted(set(cat_cols))
num_cols = sorted(set(num_cols))

X = X.reindex(sorted(X.columns), axis=1)
X_test = X_test.reindex(sorted(X_test.columns), axis=1)

# In shared002, te_names was defined in RealMLP cell.
# Because this notebook extracts only LGB, define it explicitly here.
te_names = [f"_{c}TE_class{cls}" for c in combo_names for cls in range(3)]

print("X:", X.shape)
print("X_test:", X_test.shape)
print("cat_cols:", len(cat_cols), cat_cols)
print("num_cols:", len(num_cols), num_cols)
print("combo_names:", combo_names)
print("te_names:", te_names)
display(X.head())

X: (577347, 24)
X_test: (247435, 24)
cat_cols: 14 ['alpha_cat_', 'alpha_cat__delta_cat__', 'delta_100_quantile_bin_', 'delta_500_quantile_bin_', 'delta_cat_', 'g_cat_', 'galaxy_population', 'i_cat_', 'r_cat_', 'redshift_cat_', 'spectral_type', 'u_cat_', 'u_cat__z_cat__', 'z_cat_']
num_cols: 10 ['_g_/_redshift', '_i_/_redshift', 'alpha', 'delta', 'g', 'i', 'r', 'redshift', 'u', 'z']
combo_names: ['alpha_cat__delta_cat__', 'u_cat__z_cat__']
te_names: ['_alpha_cat__delta_cat__TE_class0', '_alpha_cat__delta_cat__TE_class1', '_alpha_cat__delta_cat__TE_class2', '_u_cat__z_cat__TE_class0', '_u_cat__z_cat__TE_class1', '_u_cat__z_cat__TE_class2']


,_g_/_redshift,_i_/_redshift,alpha,alpha_cat_,alpha_cat__delta_cat__,delta,delta_100_quantile_bin_,delta_500_quantile_bin_,delta_cat_,g,...,r,r_cat_,redshift,redshift_cat_,spectral_type,u,u_cat_,u_cat__z_cat__,z,z_cat_
0,53.536564,47.085331,147.734256,0,0,16.959273,43,215,0,21.895559,...,20.357926,0,0.408982,0,0,25.472123,0,0,18.621057,0
1,120.821922,109.041748,127.988677,1,1,32.346716,66,331,1,19.087062,...,17.587208,1,0.157976,0,0,20.778509,1,1,16.786433,1
2,7.464886,7.289058,179.792648,2,2,35.344843,71,358,2,21.079128,...,21.171840,2,2.823770,1,1,21.035203,2,2,20.557366,2
3,39.266460,34.257919,225.818295,3,3,48.569421,91,457,3,21.050736,...,19.017754,3,0.536099,0,0,23.305056,3,3,17.914952,3
4,35.035980,32.207016,141.836135,4,4,19.342852,46,233,4,19.471680,...,18.234449,4,0.555761,0,0,21.703158,2,4,17.616185,3


In [4]:
# ============================================================
# 3. LightGBM params from shared002
# ============================================================

lgb_params = {
    "objective": "multiclass",
    "num_class": 3,
    "n_estimators": 3000,
    "learning_rate": 0.03,
    "max_depth": 10,
    "num_leaves": 127,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "class_weight": "balanced",
    "random_state": SEED,
    "device": "cpu",
    "n_jobs": -1,
    "verbose": -1,
}

print(json.dumps(lgb_params, indent=2))

{
  "objective": "multiclass",
  "num_class": 3,
  "n_estimators": 3000,
  "learning_rate": 0.03,
  "max_depth": 10,
  "num_leaves": 127,
  "subsample": 0.8,
  "colsample_bytree": 0.8,
  "class_weight": "balanced",
  "random_state": 42,
  "device": "cpu",
  "n_jobs": -1,
  "verbose": -1
}


In [5]:
# ============================================================
# 4. Train LightGBM only
# ============================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

oof_lgb = np.zeros((len(X), 3), dtype=np.float32)
test_lgb = np.zeros((len(X_test), 3), dtype=np.float32)

fold_rows = []
feature_importance_rows = []
models = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- Fold {fold} [shared002 LightGBM only] ---")

    X_tr = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    X_tst = X_test.copy()

    y_tr = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]

    # Fold-safe TargetEncoder:
    # fit only on train fold, transform valid/test.
    encoder = TargetEncoder(cv=FOLDS, smooth="auto", shuffle=True, random_state=SEED)
    tr_enc = encoder.fit_transform(X_tr[combo_names], y_tr)
    val_enc = encoder.transform(X_val[combo_names])
    tst_enc = encoder.transform(X_tst[combo_names])

    X_tr[te_names] = tr_enc
    X_val[te_names] = val_enc
    X_tst[te_names] = tst_enc

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(150, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )

    best_iter = model.best_iteration_ or lgb_params["n_estimators"]

    val_proba = model.predict_proba(X_val, num_iteration=best_iter)
    test_proba = model.predict_proba(X_tst, num_iteration=best_iter)

    oof_lgb[val_idx] = val_proba.astype(np.float32)
    test_lgb += test_proba.astype(np.float32) / FOLDS

    fold_score = metric(y_val, oof_lgb[val_idx])
    print(f"Fold {fold} LGB Score: {fold_score:.8f}")
    print("best_iteration:", best_iter)
    print(confusion_matrix(y_val, np.argmax(oof_lgb[val_idx], axis=1)))

    fold_rows.append({
        "fold": fold,
        "balanced_accuracy": float(fold_score),
        "best_iteration": int(best_iter),
        "n_train": int(len(tr_idx)),
        "n_valid": int(len(val_idx)),
    })

    fi = pd.DataFrame({
        "feature": X_tr.columns.tolist(),
        "importance_gain": model.booster_.feature_importance(importance_type="gain"),
        "importance_split": model.booster_.feature_importance(importance_type="split"),
        "fold": fold,
    })
    feature_importance_rows.append(fi)
    models.append(model)

lgb_score = metric(y, oof_lgb)
fold_scores = pd.DataFrame(fold_rows)
feature_importance = pd.concat(feature_importance_rows, ignore_index=True)

print(f"\n>>> shared002 LightGBM Overall OOF Score: {lgb_score:.8f}")
display(fold_scores)

oof_pred = np.argmax(oof_lgb, axis=1)
print(classification_report(y, oof_pred, target_names=CLASS_NAMES))


--- Fold 1 [shared002 LightGBM only] ---
[200]	valid_0's multi_logloss: 0.104423
[400]	valid_0's multi_logloss: 0.0996431
Fold 1 LGB Score: 0.96259758
best_iteration: 413
[[73168   886  1442]
 [  509 22654   266]
 [  724    75 15746]]

--- Fold 2 [shared002 LightGBM only] ---
[200]	valid_0's multi_logloss: 0.105866
[400]	valid_0's multi_logloss: 0.101002
Fold 2 LGB Score: 0.96290072
best_iteration: 401
[[73042   876  1578]
 [  458 22696   275]
 [  696    90 15759]]

--- Fold 3 [shared002 LightGBM only] ---
[200]	valid_0's multi_logloss: 0.108038
[400]	valid_0's multi_logloss: 0.103526
Fold 3 LGB Score: 0.96178188
best_iteration: 395
[[72962   912  1622]
 [  482 22655   292]
 [  712    83 15749]]

--- Fold 4 [shared002 LightGBM only] ---
[200]	valid_0's multi_logloss: 0.106919
[400]	valid_0's multi_logloss: 0.102255
Fold 4 LGB Score: 0.96216121
best_iteration: 409
[[73080   898  1518]
 [  511 22661   256]
 [  738    69 15738]]

--- Fold 5 [shared002 LightGBM only] ---
[200]	valid_0's m

,fold,balanced_accuracy,best_iteration,n_train,n_valid
0,1,0.962598,413,461877,115470
1,2,0.962901,401,461877,115470
2,3,0.961782,395,461878,115469
3,4,0.962161,409,461878,115469
4,5,0.963123,395,461878,115469


              precision    recall  f1-score   support

      GALAXY       0.98      0.97      0.98    377480
         QSO       0.96      0.97      0.96    117143
        STAR       0.90      0.95      0.92     82724

    accuracy                           0.97    577347
   macro avg       0.95      0.96      0.95    577347
weighted avg       0.97      0.97      0.97    577347



In [6]:
# ============================================================
# 5. Submission
# ============================================================

test_pred = np.argmax(test_lgb, axis=1)

submission = pd.DataFrame({
    ID: test_id,
    TARGET: test_pred,
})
submission[TARGET] = submission[TARGET].map(ID_TO_CLASS)

assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID].equals(sample[ID])

print(submission[TARGET].value_counts(normalize=True))
display(submission.head())

submission_path = OUTDIR / "submission_shared002_lgb_as_is.csv"
submission.to_csv(submission_path, index=False)

print("saved:", submission_path)

class
GALAXY    0.643785
QSO       0.204518
STAR      0.151696
Name: proportion, dtype: float64


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


saved: /kaggle/working/exp_20260604_010_shared002_lgb_as_is/submission_shared002_lgb_as_is.csv


In [7]:
# ============================================================
# 6. Save artifacts
# ============================================================

# Required npy filenames include exp_* id.
np.save(OOF_NPY, oof_lgb.astype(np.float32))
np.save(PRED_NPY, test_lgb.astype(np.float32))

oof_df = pd.DataFrame({
    ID: train_id if "train_id" in globals() else np.arange(len(y)),
    "y_true_id": y.values,
    "y_true": pd.Series(y.values).map(ID_TO_CLASS).values,
    "y_pred_id": oof_pred,
    "y_pred": pd.Series(oof_pred).map(ID_TO_CLASS).values,
})
for cls_id, cls_name in ID_TO_CLASS.items():
    oof_df[f"proba_{cls_name}"] = oof_lgb[:, cls_id]
oof_df.to_csv(OUTDIR / "oof_shared002_lgb_as_is.csv", index=False)

pred_df = pd.DataFrame({
    ID: test_id,
    "pred_id": test_pred,
    "pred_label": pd.Series(test_pred).map(ID_TO_CLASS).values,
})
for cls_id, cls_name in ID_TO_CLASS.items():
    pred_df[f"proba_{cls_name}"] = test_lgb[:, cls_id]
pred_df.to_csv(OUTDIR / "pred_shared002_lgb_as_is.csv", index=False)

fold_scores.to_csv(OUTDIR / "fold_scores.csv", index=False)

feature_importance_summary = (
    feature_importance
    .groupby("feature", as_index=False)[["importance_gain", "importance_split"]]
    .mean()
    .sort_values("importance_gain", ascending=False)
)
feature_importance.to_csv(OUTDIR / "feature_importance_by_fold.csv", index=False)
feature_importance_summary.to_csv(OUTDIR / "feature_importance.csv", index=False)

feature_info = {
    "base_cat_cols_initial": cat_cols,
    "new_cat_cols": new_cat_cols,
    "new_num_cols": new_num_cols,
    "combo_names": combo_names,
    "te_names": te_names,
    "n_features_before_te": int(X.shape[1]),
    "n_features_after_te": int(X.shape[1] + len(te_names)),
}
save_json(feature_info, OUTDIR / "feature_info.json")

cv_result = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "model": "LightGBM",
    "source": "shared002_lgb_part_only",
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "cv_score": float(lgb_score),
    "fold_scores": fold_scores.to_dict(orient="records"),
    "class_names": CLASS_NAMES,
    "label_mapping": CLASS_TO_ID,
    "params": lgb_params,
    "features": feature_info,
    "use_original": False,
    "use_fe": True,
    "fe_family": "shared002_lgb_fe_as_is",
    "use_bias_search": False,
    "submission_path": str(submission_path),
    "oof_path": str(OOF_NPY),
    "pred_path": str(PRED_NPY),
}
save_json(cv_result, OUTDIR / "cv_result.json")

print("Saved artifacts:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Saved artifacts:
 - cv_result.json
 - feature_importance.csv
 - feature_importance_by_fold.csv
 - feature_info.json
 - fold_scores.csv
 - oof_exp_20260604_010_shared002_lgb_as_is_proba.npy
 - oof_shared002_lgb_as_is.csv
 - pred_exp_20260604_010_shared002_lgb_as_is_proba.npy
 - pred_shared002_lgb_as_is.csv
 - submission_shared002_lgb_as_is.csv


In [8]:
# ============================================================
# 7. Update oof_registry.csv
# ============================================================

registry_path = WORKDIR / "oof_registry.csv"

registry_row = {
    "exp_id": EXP_ID,
    "model_family": "LightGBM",
    "feature_family": "shared002_lgb_fe_as_is",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(lgb_score),
    "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
    "public_lb": np.nan,
    "use_original": False,
    "use_fe": True,
    "use_bias_search": False,
    "oof_path": str(OOF_NPY),
    "pred_path": str(PRED_NPY),
    "submission_path": str(submission_path),
    "role": "shared_code_lgb_material_candidate",
    "status": "completed",
    "keep_hold_drop": "pending_public_lb_and_corr",
    "notes": "shared002 LightGBM part only. RealMLP and Hill Climb are not run. TargetEncoder is fold-safe.",
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

display(registry.tail())
print("registry saved:", registry_path)

,exp_id,model_family,feature_family,cv_metric,cv_score,fold_std,public_lb,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260604_010_shared002_lgb_as_is,LightGBM,shared002_lgb_fe_as_is,balanced_accuracy_score,0.962513,0.000487,NaN,False,True,False,/kaggle/working/exp_20260604_010_shared002_lgb...,/kaggle/working/exp_20260604_010_shared002_lgb...,/kaggle/working/exp_20260604_010_shared002_lgb...,shared_code_lgb_material_candidate,completed,pending_public_lb_and_corr,shared002 LightGBM part only. RealMLP and Hill...


registry saved: /kaggle/working/oof_registry.csv


In [9]:
# ============================================================
# 8. memo.yml
# ============================================================

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "shared002 LightGBM part as-is",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "LightGBM",
        "created_at": "2026-06-04",
    },
    "objective": {
        "summary": (
            "Extract and reproduce only the LightGBM part from shared002. "
            "Do not run RealMLP or Hill Climbing. "
            "Use shared002 feature engineering and fold-safe TargetEncoder. "
            "Save OOF/pred npy files with exp_id in filename for later blend/correlation."
        ),
        "success_criteria": [
            "run shared002 LGB part only",
            "save OOF proba npy with exp_id in filename",
            "save test pred proba npy with exp_id in filename",
            "save submission",
            "save cv_result",
            "save fold_scores",
            "save feature_importance",
            "update oof_registry",
        ],
    },
    "data": {
        "train_path": str(TRAIN_PATH),
        "test_path": str(TEST_PATH),
        "sample_submission_path": str(SAMPLE_SUB_PATH),
        "target_col": TARGET,
        "id_col": ID,
        "use_original": False,
    },
    "features": {
        "feature_family": "shared002_lgb_fe_as_is",
        "ratio_features": [
            "_g_/_redshift",
            "_i_/_redshift",
        ],
        "floor_category_features": "all numeric columns converted by floor() to *_cat_",
        "bin_features": [
            "delta_100_quantile_bin_",
            "delta_500_quantile_bin_",
        ],
        "combo_features": combo_names,
        "target_encoded_combo_features": te_names,
        "target_encoder_safety": (
            "TargetEncoder is fit inside each fold on X_tr[combo_names], y_tr, "
            "then transforms X_val and X_tst."
        ),
    },
    "cv": {
        "scheme": "StratifiedKFold",
        "n_splits": FOLDS,
        "shuffle": True,
        "random_state": SEED,
        "score": float(lgb_score),
        "fold_scores": fold_scores.to_dict(orient="records"),
        "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
    },
    "params": lgb_params,
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_csv": "oof_shared002_lgb_as_is.csv",
        "pred_csv": "pred_shared002_lgb_as_is.csv",
        "submission": "submission_shared002_lgb_as_is.csv",
        "cv_result": "cv_result.json",
        "fold_scores": "fold_scores.csv",
        "feature_importance": "feature_importance.csv",
        "feature_info": "feature_info.json",
        "registry": "oof_registry.csv",
    },
    "judgement": {
        "status": "pending_public_lb_and_corr",
        "reason": (
            "This is a shared-code-derived LGB FE material. "
            "Its value depends on CV, Public LB, and OOF correlation against 007."
        ),
        "next_action": [
            "Submit submission_shared002_lgb_as_is.csv",
            "Record Public LB",
            "Add OOF/pred npy to npy-files dataset",
            "Compare OOF correlation against 007_lgb002_bias",
            "Blend 007 + 010 if correlation is not too high",
            "If CV is high but Public LB drops, classify as risky_cv_only",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("memo saved:", memo_path)

memo saved: /kaggle/working/exp_20260604_010_shared002_lgb_as_is/memo.yml
